# Chapter 5 — Classes & Objects

**Time:** ~2 hours  
**Goal:** Understand classes and objects. Every file in the project defines one class. This is the most important chapter.

---

## 5.1 Why Classes?

As programs grow, you end up with many related variables and functions. For example, to simulate a sensor you might have:

```python
sensor_temperature = 32
sensor_humidity = 65.4
sensor_is_occupied = True

def generate_temperature(): ...
def generate_humidity(): ...
def generate_occupancy(): ...
```

What if you have 10 sensors? You need 30 variables and 30 functions, all named differently.

A **class** solves this by bundling related data (variables) and behavior (functions) into a single, reusable unit. You define it once and create as many **instances** (copies) as you need.

## 5.2 Defining a Class — The Simplest Case

```python
class ClassName:
    def __init__(self, param1, param2):
        self.attribute1 = param1
        self.attribute2 = param2
```

- `class` — keyword that starts a class definition
- `ClassName` — name of the class (use CamelCase: `SensorSimulator`, not `sensor_simulator`)
- `__init__` — special method called automatically when you create an object. It sets up the initial state.
- `self` — refers to the specific object being created. It is always the first parameter of every method.
- `self.attribute` — a variable that belongs to this object (accessible anywhere in the class)

In [ ]:
class Sensor:
    def __init__(self, location):
        self.location = location
        self.temperature = 0
        self.humidity = 0


# Create an instance (object) of the Sensor class
my_sensor = Sensor("Living Room")

print(my_sensor.location)     # Living Room
print(my_sensor.temperature)  # 0
print(my_sensor.humidity)     # 0

## 5.3 Adding Methods

A **method** is a function that belongs to a class. It always has `self` as its first parameter.

In [ ]:
import random

class Sensor:
    def __init__(self, location):
        self.location = location
        self.temperature = 0
        self.humidity = 0

    def generate_reading(self):
        self.temperature = random.randint(22, 38)
        self.humidity = round(random.uniform(40, 90), 1)

    def print_reading(self):
        print(f"[{self.location}] Temp: {self.temperature}°C  Humidity: {self.humidity}%")


sensor = Sensor("Bedroom")
sensor.generate_reading()  # updates temperature and humidity
sensor.print_reading()     # prints the current values

Call `generate_reading()` again and watch the values change:

In [ ]:
for i in range(5):
    sensor.generate_reading()
    sensor.print_reading()

## 5.4 Multiple Instances

One class definition, many objects — each with its own data:

In [ ]:
sensor_bedroom = Sensor("Bedroom")
sensor_kitchen = Sensor("Kitchen")
sensor_office  = Sensor("Office")

for sensor in [sensor_bedroom, sensor_kitchen, sensor_office]:
    sensor.generate_reading()
    sensor.print_reading()

Each sensor has independent data. Changing one does not affect the others.

## 5.5 Storing State Across Method Calls

A key power of classes is that `self` attributes persist between method calls. This is how `DataLogger` accumulates records over time:

In [ ]:
class EnergyTracker:
    def __init__(self):
        self.total_energy = 0.0
        self.cycle_count = 0

    def record_cycle(self, energy_used):
        self.total_energy += energy_used
        self.cycle_count += 1

    def summary(self):
        avg = self.total_energy / self.cycle_count if self.cycle_count > 0 else 0
        print(f"Cycles: {self.cycle_count}")
        print(f"Total energy: {self.total_energy:.4f} kWh")
        print(f"Average per cycle: {avg:.4f} kWh")


tracker = EnergyTracker()
tracker.record_cycle(0.025)
tracker.record_cycle(0.0)    # AC was off
tracker.record_cycle(0.025)
tracker.record_cycle(0.025)
tracker.summary()

## 5.6 A Realistic Mini-Project

Now build three classes that work together — just like the real project:

In [ ]:
import random

class SensorSimulator:
    """Generates simulated sensor readings."""

    def __init__(self):
        self.temperature = 0
        self.humidity = 0.0
        self.is_occupied = False

    def generate(self):
        self.temperature = random.randint(22, 38)
        self.humidity = round(random.uniform(40, 90), 1)
        self.is_occupied = random.choice([True, False])


class DecisionEngine:
    """Decides whether AC should be on based on sensor data."""

    THRESHOLD = 30

    def __init__(self):
        self.ac_on = False

    def evaluate(self, temperature, is_occupied):
        if not is_occupied:
            self.ac_on = False
        elif temperature > self.THRESHOLD:
            self.ac_on = True
        else:
            self.ac_on = False


class EnergyCalculator:
    """Calculates and tracks energy consumption."""

    AC_POWER_KW = 1.5
    CYCLE_HOURS = 1 / 60

    def __init__(self):
        self.total_energy = 0.0

    def record(self, ac_on):
        if ac_on:
            energy = self.AC_POWER_KW * self.CYCLE_HOURS
            self.total_energy += energy
            return energy
        return 0.0


# Wire them together
sensor     = SensorSimulator()
engine     = DecisionEngine()
calculator = EnergyCalculator()

print(f"{'Cycle':<6} {'Temp':>5} {'Occupied':>9} {'AC':>5} {'Energy':>10}")
print("-" * 40)

for cycle in range(1, 8):
    sensor.generate()
    engine.evaluate(sensor.temperature, sensor.is_occupied)
    energy = calculator.record(engine.ac_on)

    print(f"{cycle:<6} {sensor.temperature:>4}°C {str(sensor.is_occupied):>9} {str(engine.ac_on):>5} {energy:>9.4f}")

print("-" * 40)
print(f"Total energy: {calculator.total_energy:.4f} kWh")

Run this multiple times. Each run simulates a different scenario because the sensor generates random values.

**Notice:** Each class has one job. They do not know about each other — they communicate only through the data passed in. This is the design principle behind the real project.

## 5.7 Reading the Real SensorSimulator

Now open `src/sensor.py`. You will see the real `SensorSimulator` class. Compare it to the simplified version above:

- `__init__` sets up the same attributes: temperature, humidity, occupancy
- The generate method uses `random` just like yours
- It reads constants from `config.py` instead of hardcoding values — that is the only real difference

You have already built and understood the core of this class.

---
## Exercises

**Exercise 1:** Write a `DataLogger` class with:
- `__init__` that creates an empty list called `records`
- A method `log(temperature, ac_on, energy)` that appends a dictionary to `records`
- A method `show_all()` that prints every record

Call it 5 times with different values and then call `show_all()`.

In [ ]:
# Write your answer here


**Exercise 2:** Add a method `average_temperature()` to your `DataLogger` that calculates and returns the average temperature across all logged records.

In [ ]:
# Write your answer here


**Exercise 3:** Wire all four classes together: `SensorSimulator`, `DecisionEngine`, `EnergyCalculator`, and your `DataLogger`. Run a 10-cycle simulation and at the end print the average temperature and total energy.

In [ ]:
import random
# Write your answer here


---
**Chapter 5 complete — and Part 1 is done.**

You now know everything needed to read every Python file in the project.

Move on to Chapter 6 — Project Walkthrough.